# Small Theta Results:



In [2]:
import numpy as np
import random
import pandas as pd

name = "Continuing_UM_R"
path = name + ".csv"
Res = pd.read_csv(path)

In [3]:
# Hyperparameters we selected

Continuing_UM_grid = [
   (theta, alpha, gamma, epsilon)
   for theta in [1e-9, 5e-9, 1e-8, 5e-8, 1e-7, 5e-7, 1e-6]
   for alpha in [0.0001, 0.0005, 0.001]
   for gamma in [0.7, 0.8, 0.9]
   for epsilon in [0.1, 0.2, 0.4]
   ]

seed = 309
random.seed(seed)
np.random.seed(seed)
seeds = np.random.choice(100000, size=100, replace=False).tolist()

k = 11
n = int(k*(k-1)/2)
max_it = 100000

In [4]:
grid = Continuing_UM_grid
a = [(0.0, 0.0, 0.0, 0.0, 0, 0) for (theta, alpha, gamma, epsilon) in grid]
A = pd.DataFrame(a,
                index = list(grid),
                columns=['a_d', 'r_d', 'w_b', 'n_it', 'exact', 'early'])

for (theta, alpha, gamma, epsilon) in grid:
  df = Res.loc[(Res['theta'] == theta) & (Res['alpha'] == alpha) & (Res['gamma'] == gamma) & (Res['epsilon'] == epsilon)]
  avg_a_d = np.mean(df['a_d'])
  avg_r_d = np.mean(df['r_d'])
  avg_w_b = np.mean(df['w_b'])
  avg_n_it = np.mean(df['n_it'])
  A.at[(theta, alpha, gamma, epsilon), 'a_d'] = avg_a_d
  A.at[(theta, alpha, gamma, epsilon), 'r_d'] = avg_r_d
  A.at[(theta, alpha, gamma, epsilon), 'w_b'] = avg_w_b
  A.at[(theta, alpha, gamma, epsilon), 'n_it'] = avg_n_it
  A.at[(theta, alpha, gamma, epsilon), 'exact'] = (df.loc[df['w_b'] == 0]).shape[0]
  A.at[(theta, alpha, gamma, epsilon), 'early'] = (df.loc[df['n_it'] < max_it]).shape[0]

In [5]:
df = A[['r_d', 'w_b', 'exact', 'early', 'n_it']]
q = A['r_d']
min = np.min(q)

print("--Results for "+name+"--")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
  print("\n-Parameters within 1% of the actual minimum:\n")
  print(df.loc[q < 0.01])

--Results for Continuing_UM_R--

-Parameters within 1% of the actual minimum:

                                r_d   w_b  exact  early       n_it
(1e-09, 0.0001, 0.7, 0.1)  0.001907  1.00     34      0  100000.00
(1e-09, 0.0001, 0.7, 0.2)  0.002023  1.07     37      0  100000.00
(1e-09, 0.0001, 0.7, 0.4)  0.002809  1.27     32      0  100000.00
(1e-09, 0.0001, 0.8, 0.1)  0.002320  1.08     30      0  100000.00
(1e-09, 0.0001, 0.8, 0.2)  0.001107  0.85     36      0  100000.00
(1e-09, 0.0001, 0.8, 0.4)  0.001692  1.03     39      0  100000.00
(1e-09, 0.0001, 0.9, 0.2)  0.002366  1.01     36      0  100000.00
(1e-09, 0.0001, 0.9, 0.4)  0.001233  0.93     40      0  100000.00
(1e-09, 0.0005, 0.7, 0.1)  0.004044  1.57     19      0  100000.00
(1e-09, 0.0005, 0.7, 0.2)  0.006004  1.81     17      0  100000.00
(1e-09, 0.0005, 0.7, 0.4)  0.009658  2.23     10      0  100000.00
(1e-09, 0.0005, 0.8, 0.1)  0.002177  1.11     32      0  100000.00
(1e-09, 0.0005, 0.8, 0.2)  0.003446  1.41     23  

In [6]:
bh = round(df.loc[q - min < 0.0001], 6)
bh

,r_d,w_b,exact,early,n_it
"(1e-09, 0.0005, 0.9, 0.1)",0.000800,0.73,43,0,100000.00
"(5e-09, 0.0005, 0.9, 0.1)",0.000800,0.73,43,0,100000.00
"(1e-08, 0.0005, 0.9, 0.1)",0.000800,0.73,43,0,100000.00
"(5e-08, 0.0005, 0.9, 0.1)",0.000800,0.73,43,0,100000.00
"(1e-07, 0.0005, 0.9, 0.1)",0.000800,0.73,43,0,100000.00
"(5e-07, 0.0005, 0.9, 0.1)",0.000809,0.74,41,68,84778.96


In [7]:
print('Hyperparameter combinations that converge early less than 5 times:')
print((df.loc[df['early'] == 0]).shape[0])
print('\nHyperparameter combinations that converge early 5 times or less:')
print((df.loc[df['early'] <= 5]).shape[0])
print('\nHyperparameter combinations that converge early at least 95 times:')
print((df.loc[df['early'] >= 95]).shape[0])

Hyperparameter combinations that converge early less than 5 times:
140

Hyperparameter combinations that converge early 5 times or less:
149

Hyperparameter combinations that converge early at least 95 times:
19


In [8]:
Thetas = [1e-9, 5e-9, 1e-8, 5e-8, 1e-7, 5e-7, 1e-6]
few_early = []
most_early = []
T = pd.DataFrame(
    np.zeros([len(Thetas), 2]),
    index = Thetas,
    columns = ['<5 early', '>95 early']
)

for i in range(len(Thetas)):
  df_theta=df.iloc[(27*i):(27*(i+1))]
  T.at[Thetas[i], '<5 early'] = (np.mean(df_theta.loc[df_theta['early'] < 5, 'r_d']))
  T.at[Thetas[i], '>95 early'] = (np.mean(df_theta.loc[df_theta['early'] > 95, 'r_d']))

In [9]:
print(T)

              <5 early  >95 early
1.000000e-09  0.005860        NaN
5.000000e-09  0.005860        NaN
1.000000e-08  0.005860        NaN
5.000000e-08  0.005858        NaN
1.000000e-07  0.005854        NaN
5.000000e-07  0.007034   0.009476
1.000000e-06  0.008139   0.104909
